# Task C — Main supervised-learning project: Attack classification on the Cyber Security Attacks dataset

This notebook builds a **multiclass attack-classification** solution using the target **`Attack Type`**.

The notebook is organized to satisfy the requested checkpoints:

- clear target definition and justification
- leakage-safe preprocessing and split logic
- visible hyperparameter tuning
- comparison of **at least three models**
- metric justification and concise interpretation
- **C1** leakage-safe pipeline trace
- **C2** debugging a flawed workflow
- **C3** cost-sensitive threshold design
- **C4** failure analysis and improvement plan

> **Why multiclass?**  
> The dataset already contains labeled attack categories under `Attack Type`, so a multiclass setup is the most direct and information-preserving formulation.  
> For the threshold-design section, the notebook converts the best multiclass model into a **one-vs-rest operational decision** for a single critical class (`Malware`) because thresholding is most meaningful in a binary decision setting.

## Task / Question
**Import the required libraries, set a clean plotting style, and configure pandas output so the report tables are easy to read.**

In [ ]:

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)
from sklearn.model_selection import ParameterGrid, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Task / Question
**Load the uploaded dataset, inspect its shape and column names, and run quick sanity checks such as duplicate count and target availability.**

In [ ]:

candidate_paths = [
    Path("cybersecurity_attacks.csv"),
    Path("/mnt/data/cybersecurity_attacks.csv")
]

for path in candidate_paths:
    if path.exists():
        DATA_PATH = path
        break
else:
    raise FileNotFoundError("Could not find 'cybersecurity_attacks.csv'. Put the CSV next to the notebook or update DATA_PATH.")

df_raw = pd.read_csv(DATA_PATH)

overview = pd.DataFrame({
    "Property": ["Rows", "Columns", "Duplicate rows", "Missing target labels", "Target column"],
    "Value": [
        len(df_raw),
        df_raw.shape[1],
        int(df_raw.duplicated().sum()),
        int(df_raw["Attack Type"].isna().sum()) if "Attack Type" in df_raw.columns else "Target not found",
        "Attack Type"
    ]
})

display(overview)
display(pd.DataFrame({"Columns": df_raw.columns}))
display(df_raw.head())

## Task / Question
**State the target definition and make the feature-handling choices visible. Show which columns are kept, excluded for high cardinality / weak generalization, or excluded because they may leak post-event information.**

In [ ]:

feature_decisions = pd.DataFrame([
    ["Attack Type", "Target", "Multiclass label to predict."],
    ["Timestamp", "Engineer then drop raw", "Use hour/day/month/year features; raw timestamp is not used directly."],
    ["Source Port", "Keep", "Core traffic metadata available at or before classification time."],
    ["Destination Port", "Keep", "Core traffic metadata available at or before classification time."],
    ["Protocol", "Keep", "Low-cardinality protocol feature; operationally realistic."],
    ["Packet Length", "Keep", "Packet-size information can be informative and is available early."],
    ["Packet Type", "Keep", "Compact categorical network descriptor."],
    ["Traffic Type", "Keep", "Compact categorical network/application traffic descriptor."],
    ["Anomaly Scores", "Keep", "Retained as an upstream signal, but interpreted carefully."],
    ["Network Segment", "Keep", "Operational context that may be known at ingestion time."],
    ["Log Source", "Keep", "Operational context available at event collection time."],
    ["Source IP Address", "Drop", "Very high cardinality; raw IPs often overfit. Safer future option: subnet or ASN engineering."],
    ["Destination IP Address", "Drop", "Very high cardinality; raw IPs often overfit."],
    ["Payload Data", "Drop", "Extremely high cardinality / near-identifier style content; weak baseline generalization."],
    ["User Information", "Drop", "Very high cardinality and not a robust first-pass feature."],
    ["Device Information", "Drop", "Very high cardinality and not a robust first-pass feature."],
    ["Geo-location Data", "Drop", "High cardinality; would need safer aggregation rather than raw strings."],
    ["Proxy Information", "Drop", "High missingness and high cardinality."],
    ["Malware Indicators", "Drop (leakage risk)", "Likely reflects downstream inspection findings rather than raw input evidence."],
    ["Alerts/Warnings", "Drop (leakage risk)", "Alert outcomes may be generated after detection logic fires."],
    ["Attack Signature", "Drop (leakage risk)", "A signature is often almost equivalent to a labeled security finding."],
    ["Action Taken", "Drop (leakage risk)", "Response action is clearly post-decision information."],
    ["Severity Level", "Drop (leakage risk)", "Severity is typically assigned after or alongside detection."],
    ["Firewall Logs", "Drop (leakage risk)", "Potential downstream logging artifact rather than raw predictive input."],
    ["IDS/IPS Alerts", "Drop (leakage risk)", "Potential direct post-detection artifact."]
], columns=["Column", "Decision", "Reason"])

display(feature_decisions)

## C1 — Task / Question
**Provide one compact pseudocode block or step list that shows the full preprocessing-and-model pipeline.**

```text
1. Load the CSV dataset.
2. Remove exact duplicate rows BEFORE any split.
3. Parse Timestamp and engineer Hour, DayOfWeek, Month, and Year.
4. Define the multiclass target = Attack Type.
5. Exclude raw identifiers / high-cardinality text fields and exclude leakage-prone post-event fields.
6. Split the data with stratification into Train / Validation / Test.
7. Fit imputers, encoders, and scalers using TRAIN only inside each candidate pipeline.
8. Tune model hyperparameters using TRAIN and compare models on VALIDATION.
9. Select the best model using macro-F1 (primary) and balanced accuracy (secondary).
10. Refit the chosen configuration on TRAIN + VALIDATION and evaluate once on TEST.
11. For cost-sensitive thresholding, use VALIDATION probabilities for Malware-vs-rest threshold selection.
12. Freeze the chosen threshold and apply it once to TEST.
13. Inspect misclassified TEST cases and group failure modes into actionable categories.
```

## C1 — Task / Question
**Create a stage-by-stage leakage-safe trace table with the columns: Stage, Input, Output, Fitted on which data?, and Why here? Include one final row called _Leakage check_.**

In [ ]:

pipeline_trace = pd.DataFrame([
    ["Load data", "CSV file", "Raw dataframe", "No fitting", "Start from the original data source."],
    ["Remove duplicates", "Raw dataframe", "Deduplicated dataframe", "No fitting", "Prevent the same event appearing in more than one split."],
    ["Parse time", "Deduplicated dataframe", "Timestamp-derived features", "No fitting", "Create usable time features without learning from labels."],
    ["Feature curation", "Parsed dataframe", "Leakage-safe candidate features", "Design choice before modelling", "Remove raw identifiers and post-event columns."],
    ["Split", "Curated features + target", "Train / Validation / Test", "No fitting", "Create an untouched test set early."],
    ["Impute numerics", "Train numeric columns", "Completed numeric matrix", "Train only", "Avoid using validation/test statistics."],
    ["Scale numerics", "Imputed train numerics", "Scaled numeric matrix", "Train only", "Keep linear model training stable."],
    ["Impute categoricals", "Train categorical columns", "Completed categorical matrix", "Train only", "Avoid leaking category frequencies."],
    ["Encode categoricals", "Imputed train categoricals", "Model-ready categorical matrix", "Train only", "Handle unseen categories safely at inference."],
    ["Tune models", "Train split after pipeline transforms", "Best configuration per model family", "Train only", "Choose hyperparameters without touching test."],
    ["Select best model", "Validation predictions", "Chosen model family + settings", "Validation only", "Model comparison stays outside the test split."],
    ["Final fit", "Train + Validation", "Frozen final model", "Train + Validation only", "Use all non-test data after selection."],
    ["Final test", "Frozen final model + Test", "Unbiased generalization estimate", "No fitting on test", "Test is used exactly once at the end."],
    ["Leakage check", "Optional log / response fields + all data", "Leakage-safe design", "Design choice before modelling", "Prevents post-incident fields (e.g., Attack Signature, Action Taken, IDS/IPS Alerts) from leaking the target."]
], columns=["Stage", "Input", "Output", "Fitted on which data?", "Why here?"])

display(pipeline_trace)

## C1 — Explanation
If fitted transformations are applied in the wrong order or on the wrong split, the evaluation becomes optimistic and misleading.  
For example, scaling or imputing on the full dataset allows information from the validation or test set to influence the training representation.  
Similarly, using post-event columns such as `Action Taken` or `Attack Signature` would make the model appear stronger than it really is because it would be learning from information that may only exist **after** the attack has already been interpreted.

## Task / Question
**Prepare the modelling dataset: remove duplicates, parse timestamp, engineer time features, define the final selected feature set, and create `X` and `y`.**

In [ ]:

df = df_raw.drop_duplicates().copy()

df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")

df["Hour"] = df["Timestamp"].dt.hour
df["DayOfWeek"] = df["Timestamp"].dt.dayofweek
df["Month"] = df["Timestamp"].dt.month
df["Year"] = df["Timestamp"].dt.year

TARGET = "Attack Type"

selected_features = [
    "Source Port",
    "Destination Port",
    "Protocol",
    "Packet Length",
    "Packet Type",
    "Traffic Type",
    "Anomaly Scores",
    "Network Segment",
    "Log Source",
    "Hour",
    "DayOfWeek",
    "Month",
    "Year"
]

X = df[selected_features].copy()
y = df[TARGET].copy()

prep_summary = pd.DataFrame({
    "Item": ["Rows after duplicate removal", "Selected feature count", "Target classes"],
    "Value": [len(df), len(selected_features), y.nunique()]
})

display(prep_summary)
display(pd.DataFrame({"Selected features": selected_features}))
display(y.value_counts().rename_axis("Class").reset_index(name="Count"))

## Task / Question
**Create leakage-safe train / validation / test splits using stratification, and verify the split sizes and class balance.**

In [ ]:

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.25,
    stratify=y_train_val,
    random_state=RANDOM_STATE
)

split_overview = pd.DataFrame({
    "Split": ["Train", "Validation", "Test"],
    "Rows": [len(X_train), len(X_val), len(X_test)],
    "Fraction of full data": [len(X_train)/len(X), len(X_val)/len(X), len(X_test)/len(X)]
})

class_balance = pd.concat([
    y_train.value_counts(normalize=True).rename("Train"),
    y_val.value_counts(normalize=True).rename("Validation"),
    y_test.value_counts(normalize=True).rename("Test")
], axis=1).round(4)

display(split_overview)
display(class_balance)

## Task / Question
**Define the preprocessing pipelines and at least three candidate models. Keep the code visible and concise.**

In [ ]:

categorical_features = [c for c in selected_features if X[c].dtype == "object"]
numeric_features = [c for c in selected_features if c not in categorical_features]

linear_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features)
    ]
)

tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
        ]), categorical_features)
    ]
)

model_spaces = {
    "SGD Logistic": {
        "preprocessor": linear_preprocessor,
        "base_model": SGDClassifier(
            loss="log_loss",
            penalty="l2",
            max_iter=1500,
            tol=1e-3,
            random_state=RANDOM_STATE
        ),
        "param_grid": {
            "alpha": [1e-4, 1e-3, 1e-2]
        }
    },
    "Decision Tree": {
        "preprocessor": tree_preprocessor,
        "base_model": DecisionTreeClassifier(random_state=RANDOM_STATE),
        "param_grid": {
            "max_depth": [6, 12, None],
            "min_samples_leaf": [1, 5, 20]
        }
    },
    "Random Forest": {
        "preprocessor": tree_preprocessor,
        "base_model": RandomForestClassifier(
            n_estimators=120,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "param_grid": {
            "max_depth": [8, 14, None],
            "min_samples_leaf": [1, 5]
        }
    }
}

display(pd.DataFrame({
    "Model": list(model_spaces.keys()),
    "Why included": [
        "Fast linear probabilistic baseline with visible regularization.",
        "Simple non-linear tree baseline that is easy to inspect.",
        "Stronger ensemble baseline that usually improves stability."
    ]
}))

## Task / Question
**Tune the candidate models using the training split only. Show visible evidence by storing every tried hyperparameter setting and its validation metrics.**

In [ ]:

def build_pipeline(preprocessor, model):
    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

tuning_rows = []
best_models = {}

for model_name, config in model_spaces.items():
    best_score = -np.inf
    best_pipeline = None
    best_params = None

    for params in ParameterGrid(config["param_grid"]):
        model = clone(config["base_model"]).set_params(**params)
        pipeline = build_pipeline(config["preprocessor"], model)

        pipeline.fit(X_train, y_train)
        val_pred = pipeline.predict(X_val)

        row = {
            "Model": model_name,
            "Hyperparameters": params,
            "Validation Accuracy": accuracy_score(y_val, val_pred),
            "Validation Balanced Accuracy": balanced_accuracy_score(y_val, val_pred),
            "Validation Macro F1": f1_score(y_val, val_pred, average="macro")
        }
        tuning_rows.append(row)

        if row["Validation Macro F1"] > best_score:
            best_score = row["Validation Macro F1"]
            best_pipeline = pipeline
            best_params = params

    best_models[model_name] = {
        "pipeline": best_pipeline,
        "best_params": best_params,
        "best_validation_macro_f1": best_score
    }

tuning_results = pd.DataFrame(tuning_rows).sort_values(
    by=["Validation Macro F1", "Validation Balanced Accuracy", "Validation Accuracy"],
    ascending=False
).reset_index(drop=True)

display(tuning_results)

## Task / Question
**Compare the tuned models on the validation set, justify the metrics, and visualize the comparison.**

**Metric justification**

- **Macro-F1** is the **primary** model-selection metric because it gives equal weight to each attack class and punishes models that perform poorly on one class even if overall accuracy is acceptable.
- **Balanced accuracy** is included to show class-wise recall fairness.
- **Accuracy** is reported as a familiar secondary measure, but it is not used alone because it can hide uneven per-class performance.

In [ ]:

model_comparison_rows = []

for model_name, info in best_models.items():
    model = info["pipeline"]
    val_pred = model.predict(X_val)

    model_comparison_rows.append({
        "Model": model_name,
        "Best Hyperparameters": info["best_params"],
        "Validation Accuracy": accuracy_score(y_val, val_pred),
        "Validation Balanced Accuracy": balanced_accuracy_score(y_val, val_pred),
        "Validation Macro F1": f1_score(y_val, val_pred, average="macro")
    })

model_comparison = pd.DataFrame(model_comparison_rows).sort_values(
    by="Validation Macro F1",
    ascending=False
).reset_index(drop=True)

display(model_comparison)

plot_df = model_comparison.melt(
    id_vars=["Model"],
    value_vars=["Validation Accuracy", "Validation Balanced Accuracy", "Validation Macro F1"],
    var_name="Metric",
    value_name="Score"
)

plt.figure(figsize=(10, 5))
sns.barplot(data=plot_df, x="Model", y="Score", hue="Metric", palette="Set2")
plt.ylim(0, 1)
plt.title("Validation-set comparison of tuned models")
plt.ylabel("Score")
plt.xlabel("")
plt.legend(title="")
plt.tight_layout()
plt.show()

best_model_name = model_comparison.loc[0, "Model"]
best_train_model = best_models[best_model_name]["pipeline"]

print(f"Best validation model: {best_model_name}")
print(f"Best hyperparameters: {best_models[best_model_name]['best_params']}")

## Concise interpretation
The table and plot above show the visible evidence used for model selection.  
Choose the model with the **highest validation macro-F1**, then use balanced accuracy as a secondary check.  
If all models are close to one another, that is still an important result because it suggests the leakage-safe feature set contains limited separable signal.

## Task / Question
**Inspect the best validation model more closely using a confusion matrix and a full class-wise report.**

In [ ]:

val_pred_best = best_train_model.predict(X_val)
label_order = sorted(y.unique())

val_cm = confusion_matrix(y_val, val_pred_best, labels=label_order)
val_report = pd.DataFrame(
    classification_report(y_val, val_pred_best, labels=label_order, output_dict=True)
).T

plt.figure(figsize=(6.5, 5))
sns.heatmap(
    val_cm,
    annot=True,
    fmt="d",
    cmap="YlGnBu",
    xticklabels=label_order,
    yticklabels=label_order
)
plt.title(f"Validation confusion matrix — {best_model_name}")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.show()

display(val_report.round(4))

## Task / Question
**Refit the chosen model configuration on Train + Validation together, then evaluate once on the untouched Test split.**

In [ ]:

winning_config = model_spaces[best_model_name]
winning_model = clone(winning_config["base_model"]).set_params(**best_models[best_model_name]["best_params"])

final_model = build_pipeline(winning_config["preprocessor"], winning_model)
final_model.fit(X_train_val, y_train_val)

test_pred = final_model.predict(X_test)

test_results = pd.DataFrame({
    "Metric": ["Accuracy", "Balanced Accuracy", "Macro F1"],
    "Test Score": [
        accuracy_score(y_test, test_pred),
        balanced_accuracy_score(y_test, test_pred),
        f1_score(y_test, test_pred, average="macro")
    ]
})

display(test_results.round(4))

test_cm = confusion_matrix(y_test, test_pred, labels=label_order)

plt.figure(figsize=(6.5, 5))
sns.heatmap(
    test_cm,
    annot=True,
    fmt="d",
    cmap="rocket_r",
    xticklabels=label_order,
    yticklabels=label_order
)
plt.title(f"Final test confusion matrix — {best_model_name}")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.show()

display(pd.DataFrame(classification_report(y_test, test_pred, labels=label_order, output_dict=True)).T.round(4))

## C2 — Task / Question
**Correct the flawed workflow and present the answer as a table with the columns: Flawed step, Why it is wrong, Correct replacement, and Correct order position. Identify at least five distinct problems.**

In [ ]:

flawed_workflow_fix = pd.DataFrame([
    [
        "Remove duplicates after the final split",
        "The same or nearly identical event could appear in more than one split, causing information leakage and inflated scores.",
        "Remove duplicates before any train/validation/test split.",
        "Immediately after loading the raw data"
    ],
    [
        "Standardize all features on the full dataset",
        "Validation and test statistics leak into training, making performance estimates too optimistic.",
        "Fit imputers/scalers on the training data only, then transform validation/test using the fitted training objects.",
        "After splitting, inside the pipeline"
    ],
    [
        "Apply SMOTE before the train/test split if necessary",
        "Synthetic samples would be influenced by observations that later appear in validation/test, contaminating evaluation.",
        "Apply SMOTE only inside the training workflow after the split, and never on validation/test.",
        "Training pipeline only, after splitting"
    ],
    [
        "Select features using test-set performance",
        "This uses the test set as a tuning device rather than an unbiased final check.",
        "Select features using training data and validation logic only.",
        "Before the final test evaluation"
    ],
    [
        "Tune the threshold on the test set",
        "Threshold tuning on the test split leaks decision calibration into the final evaluation.",
        "Tune the threshold on the validation split, then freeze it before testing.",
        "After model training, before final test use"
    ],
    [
        "Report the same test result as the final validation result",
        "Validation and test serve different purposes; merging them hides overfitting and invalidates the final estimate.",
        "Use validation for model selection and the test split exactly once for final reporting.",
        "Validation before final model freeze; test at the very end"
    ],
    [
        "Do preprocessing outside the pipeline in an uncontrolled way",
        "Manual preprocessing is easy to apply in the wrong order or on the wrong split.",
        "Wrap preprocessing and model fitting in a single reproducible pipeline.",
        "During model definition"
    ]
], columns=["Flawed step", "Why it is wrong", "Correct replacement", "Correct order position"])

display(flawed_workflow_fix)

## C3 — Task / Question
**Define a threshold-selection procedure where one false negative costs six times as much as one false positive. Compare at least five thresholds in a report table.**

Because the main problem is multiclass, thresholding is operationalized here as a **one-vs-rest decision for the critical class `Malware`**.  
This is reasonable because thresholding is most natural when we are deciding whether to **escalate a case as Malware** or not.

**Cost formula**

\[
	ext{Total Cost} = 6 	imes FN + 1 	imes FP
\]

A lower total cost is better because missing a true Malware event is assumed to be much more expensive than raising an unnecessary Malware alert.

## Task / Question
**Use validation probabilities from the best training-only model to compare at least five thresholds. Report confusion matrix, precision, recall, F1-score, balanced accuracy, and total cost.**

In [ ]:

CRITICAL_CLASS = "Malware"

val_proba = best_train_model.predict_proba(X_val)
proba_columns = list(best_train_model.named_steps["model"].classes_)
critical_index = proba_columns.index(CRITICAL_CLASS)

y_val_binary = (y_val == CRITICAL_CLASS).astype(int)
critical_scores_val = val_proba[:, critical_index]

thresholds = [0.20, 0.30, 0.40, 0.50, 0.60]
threshold_rows = []

for threshold in thresholds:
    y_pred_binary = (critical_scores_val >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_val_binary, y_pred_binary, labels=[0, 1]).ravel()
    total_cost = 6 * fn + 1 * fp

    threshold_rows.append({
        "Threshold": threshold,
        "Confusion Matrix": f"TN={tn}, FP={fp}, FN={fn}, TP={tp}",
        "Precision": precision_score(y_val_binary, y_pred_binary, zero_division=0),
        "Recall": recall_score(y_val_binary, y_pred_binary, zero_division=0),
        "F1-score": f1_score(y_val_binary, y_pred_binary, zero_division=0),
        "Balanced Accuracy": balanced_accuracy_score(y_val_binary, y_pred_binary),
        "Total Cost": total_cost
    })

threshold_report = pd.DataFrame(threshold_rows).sort_values(
    by=["Total Cost", "Recall", "F1-score"],
    ascending=[True, False, False]
).reset_index(drop=True)

display(threshold_report.round(4))

plt.figure(figsize=(8.5, 4.8))
sns.lineplot(data=threshold_report.sort_values("Threshold"), x="Threshold", y="Total Cost", marker="o", linewidth=2)
plt.title("Validation cost by Malware decision threshold")
plt.ylabel("Total Cost = 6×FN + 1×FP")
plt.tight_layout()
plt.show()

best_threshold = float(threshold_report.loc[0, "Threshold"])
print(f"Chosen validation threshold for {CRITICAL_CLASS}: {best_threshold:.2f}")

## Task / Question
**Show one short worked example for the cost calculation at one threshold.**

In [ ]:

worked_example_row = threshold_report.iloc[0].copy()
worked_threshold = worked_example_row["Threshold"]

y_pred_binary_example = (critical_scores_val >= worked_threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_val_binary, y_pred_binary_example, labels=[0, 1]).ravel()

worked_example = pd.DataFrame({
    "Item": ["Threshold", "FP", "FN", "Cost formula", "Computed total cost"],
    "Value": [
        worked_threshold,
        fp,
        fn,
        f"6 × FN + 1 × FP = 6 × {fn} + 1 × {fp}",
        6 * fn + fp
    ]
})

display(worked_example)

## Task / Question
**Freeze the chosen threshold, apply it once to the untouched test split, and explain briefly why the final threshold is reasonable.**

In [ ]:

test_proba = final_model.predict_proba(X_test)
critical_scores_test = test_proba[:, proba_columns.index(CRITICAL_CLASS)]

y_test_binary = (y_test == CRITICAL_CLASS).astype(int)
y_test_binary_pred = (critical_scores_test >= best_threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test_binary, y_test_binary_pred, labels=[0, 1]).ravel()

threshold_test_summary = pd.DataFrame({
    "Metric": ["Threshold", "TN", "FP", "FN", "TP", "Precision", "Recall", "F1-score", "Balanced Accuracy", "Total Cost"],
    "Value": [
        best_threshold,
        tn,
        fp,
        fn,
        tp,
        precision_score(y_test_binary, y_test_binary_pred, zero_division=0),
        recall_score(y_test_binary, y_test_binary_pred, zero_division=0),
        f1_score(y_test_binary, y_test_binary_pred, zero_division=0),
        balanced_accuracy_score(y_test_binary, y_test_binary_pred),
        6 * fn + fp
    ]
})

display(threshold_test_summary.round(4))

A **reasonable** final threshold is the one that best aligns with the operational cost assumption.  
Because false negatives are six times more expensive than false positives, the preferred threshold usually shifts lower than a default 0.50 if that reduces missed Malware cases enough to justify some extra false alarms.  
The validation table above provides the explicit evidence for that trade-off, and the test table shows what happened when the chosen threshold was frozen and transferred to unseen data.

## C4 — Task / Question
**Inspect at least 12 misclassified cases from the best model, group them into failure categories, and propose one corrective action for each category.**

In [ ]:

final_test_proba = final_model.predict_proba(X_test)
final_test_pred = final_model.predict(X_test)

review_df = X_test.copy()
review_df["True Label"] = y_test.values
review_df["Predicted Label"] = final_test_pred

top_two = np.sort(final_test_proba, axis=1)[:, -2:]
review_df["Top Probability"] = top_two[:, 1]
review_df["Probability Margin"] = top_two[:, 1] - top_two[:, 0]

class_to_index = {label: idx for idx, label in enumerate(final_model.named_steps["model"].classes_)}
review_df["True Label Probability"] = [
    final_test_proba[i, class_to_index[true_label]]
    for i, true_label in enumerate(review_df["True Label"])
]

train_val_df = X_train_val.copy()
train_val_df["Target"] = y_train_val.values

pattern_cols = ["Protocol", "Traffic Type", "Network Segment"]

pattern_counts = (
    train_val_df.groupby(pattern_cols)
    .size()
    .rename("Pattern Count")
    .reset_index()
)

pattern_ambiguity = (
    train_val_df.groupby(pattern_cols)["Target"]
    .nunique()
    .rename("Distinct Labels for Pattern")
    .reset_index()
)

review_df = review_df.merge(pattern_counts, on=pattern_cols, how="left")
review_df = review_df.merge(pattern_ambiguity, on=pattern_cols, how="left")

review_df["Pattern Count"] = review_df["Pattern Count"].fillna(0)
review_df["Distinct Labels for Pattern"] = review_df["Distinct Labels for Pattern"].fillna(0)

for col in categorical_features:
    seen_values = set(train_val_df[col].astype(str).dropna())
    review_df[f"Unseen::{col}"] = ~review_df[col].astype(str).isin(seen_values)

review_df["Unseen Category Count"] = review_df[[c for c in review_df.columns if c.startswith("Unseen::")]].sum(axis=1)

numeric_reference = X_train_val[numeric_features].copy()

outlier_flags = pd.DataFrame(index=review_df.index)
for col in numeric_features:
    q1 = numeric_reference[col].quantile(0.25)
    q3 = numeric_reference[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier_flags[col] = (review_df[col] < lower) | (review_df[col] > upper)

review_df["Noisy Input Flag"] = outlier_flags.any(axis=1)

def categorize_failure(row):
    if row["Distinct Labels for Pattern"] >= 3:
        return "possible label ambiguity"
    elif row["Pattern Count"] <= 10:
        return "rare-pattern failure"
    elif (row["Top Probability"] < 0.45) or (row["Probability Margin"] < 0.08):
        return "threshold effect"
    elif row["Unseen Category Count"] >= 1:
        return "weak representation"
    elif row["Noisy Input Flag"]:
        return "noisy input"
    else:
        return "class overlap"

review_df["Failure Category"] = review_df.apply(categorize_failure, axis=1)

misclassified = review_df[review_df["True Label"] != review_df["Predicted Label"]].copy()

sample_cases = (
    misclassified.groupby("Failure Category", group_keys=False)
    .head(3)
    .copy()
)

if len(sample_cases) < 12:
    remaining = misclassified.drop(sample_cases.index, errors="ignore")
    extra_needed = min(12 - len(sample_cases), len(remaining))
    if extra_needed > 0:
        sample_cases = pd.concat([
            sample_cases,
            remaining.sample(n=extra_needed, random_state=RANDOM_STATE)
        ], axis=0)

sample_cases = sample_cases.head(12)

columns_for_review = [
    "True Label", "Predicted Label", "Failure Category",
    "Top Probability", "Probability Margin", "True Label Probability",
    "Protocol", "Traffic Type", "Network Segment",
    "Source Port", "Destination Port", "Packet Length", "Anomaly Scores",
    "Pattern Count", "Distinct Labels for Pattern", "Unseen Category Count", "Noisy Input Flag"
]

display(sample_cases[columns_for_review].round(4))

## Note on the failure categories
The categories below are **heuristic diagnostic labels**, not absolute ground truth.  
Their purpose is to turn misclassifications into an actionable improvement plan rather than to claim perfect causal explanations for every error.

## Task / Question
**Create one summary table with the number of cases in each failure category and one corrective action for each category. End with a short priority paragraph.**

In [ ]:

corrective_action_map = {
    "possible label ambiguity": "Review annotation rules or relabel conflicting patterns; add label-quality checks.",
    "rare-pattern failure": "Collect more examples of rare patterns or use stronger representation learning / class-aware sampling.",
    "threshold effect": "Calibrate probabilities and retune the decision threshold on validation data.",
    "weak representation": "Engineer better features for unseen / sparse patterns (for example, subnet features instead of raw identifiers).",
    "noisy input": "Add data cleaning, robust models, or outlier-handling rules.",
    "class overlap": "Add richer domain features or sequence/context features to separate similar attack behaviors."
}

failure_summary = (
    misclassified["Failure Category"]
    .value_counts()
    .rename_axis("Failure Category")
    .reset_index(name="Number of misclassified test cases")
)

failure_summary["Corrective Action"] = failure_summary["Failure Category"].map(corrective_action_map)

display(failure_summary)

top_failure = failure_summary.iloc[0]["Failure Category"] if len(failure_summary) > 0 else "no dominant category"

print(
    "Priority recommendation: Start with the corrective action for "
    f"'{top_failure}'. This usually gives the highest return because it addresses "
    "the most frequent observed failure mode in the current test errors. "
    "After that, revisit calibration and feature engineering, especially if many "
    "errors are low-confidence or pattern-overlap cases."
)

## Final takeaway
This notebook keeps the full methodology visible:

- the target is clearly defined as **multiclass `Attack Type`**
- feature handling is justified, including **leakage-aware exclusions**
- split logic is explicit and test-safe
- tuning evidence is shown for **three model families**
- model choice is based on visible validation metrics
- threshold design is aligned with an explicit **cost function**
- failure analysis ends in an **improvement plan**

This makes the notebook suitable both as a technical submission and as a readable report of methodological choices.